# ETL y Limpieza de Datos Logísticos



In [3]:
%mamba install pandas

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.966 seconds
  Name    Version  Build                Channel
-------------------------------------------------------------
+ pandas  3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
- pip     26.2.2   pyh145f28c_0         conda-forge


In [4]:
import pandas as pd
import numpy as np

### 1. Carga de Datos y Diagnóstico Inicial

En esta etapa realizamos una exploración inicial del dataset para evaluar la estructura de las columnas, identificar tipos de datos inconsistentes, detectar valores nulos (`NaN`) y buscar posibles anomalías o registros duplicados antes de iniciar la transformación.

In [5]:
df = pd.read_csv("pedidos_logistica_raw.csv")
df.head(10)

,pedido_id,fecha_pedido,fecha_entrega,cliente,region,categoria,monto_venta,costo_envio,estado_entrega
0,ORD-10000,2026-07-03,NaN,Camila Martínez,Metropolitana,Vestuario,-24280.0,2990.0,NaN
1,ORD-10001,09/07/2026,2026-07-24,valentina fuentes,Valparaíso,Calzado,371470.0,6990.0,Devuelto
2,ORD-10002,2026-08-13,2026-08-19,Diego González,Metropolitana,Belleza,119180.0,6990.0,Entregado
3,ORD-10003,02/09/2026,2026-09-14,Tomás Silva,Región Metropolitana,Hogar,$370.400,6990.0,Entregado
4,ORD-10004,08/09/2026,NaN,Nicolás Fuentes,Araucanía,Hogar,192680.0,6990.0,En tránsito
5,ORD-10005,2026-08-21,2026-08-30,francisca díaz,Biobío,Belleza,153130.0,4990.0,Entregado
6,ORD-10006,2026-09-05,2026-09-12,Joaquín Martínez,Araucanía,Hogar,157240.0,3990.0,Entregado
7,ORD-10007,2026-08-04,NaN,Constanza Rojas,Biobío,belleza,96360.0,6990.0,En tránsito
8,ORD-10008,2026-06-10,2026-06-12,JOAQUÍN CONTRERAS,Metropolitana,Belleza,195780.0,0.0,Entregado
9,ORD-10009,2026-06-22,2026-06-25,Benjamín Rojas,RM,Vestuario,416580.0,0.0,Entregado


In [6]:
df.info()
print("Duplicados encontrados:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pedido_id       500 non-null    str    
 1   fecha_pedido    500 non-null    str    
 2   fecha_entrega   351 non-null    str    
 3   cliente         486 non-null    str    
 4   region          500 non-null    str    
 5   categoria       489 non-null    str    
 6   monto_venta     479 non-null    str    
 7   costo_envio     467 non-null    float64
 8   estado_entrega  487 non-null    str    
dtypes: float64(1), str(8)
memory usage: 19.6 KB
Duplicados encontrados: 0


### 2. Transformación y Limpieza 



In [7]:
df["monto_venta"] = (
    df["monto_venta"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(".", "", regex=False)
)
df["monto_venta"] = pd.to_numeric(df["monto_venta"], errors="coerce")
df["monto_venta"].head(10)

0    -242800.0
1    3714700.0
2    1191800.0
3     370400.0
4    1926800.0
5    1531300.0
6    1572400.0
7     963600.0
8    1957800.0
9    4165800.0
Name: monto_venta, dtype: float64

In [14]:
df["categoria"] = df["categoria"].astype(str).str.strip().str.title()
df.head()

,pedido_id,fecha_pedido,fecha_entrega,cliente,region,categoria,monto_venta,costo_envio,estado_entrega,ingreso_total,tipo_movimiento,dias_entrega,fecha_pedido_str,fecha_entrega_str
0,ORD-10000,2026-03-07,NaT,Camila Martínez,Región Metropolitana,Vestuario,-242800.0,2990.0,NaN,0.0,Devolución / Ajuste,NaN,07/03/2026,NaN
1,ORD-10001,2026-07-09,2026-07-24,Valentina Fuentes,Valparaíso,Calzado,3714700.0,6990.0,Devuelto,3721690.0,Venta,15.0,09/07/2026,24/07/2026
2,ORD-10002,2026-08-13,2026-08-19,Diego González,Región Metropolitana,Belleza,1191800.0,6990.0,Entregado,1198790.0,Venta,6.0,13/08/2026,19/08/2026
3,ORD-10003,2026-09-02,2026-09-14,Tomás Silva,Región Metropolitana,Hogar,370400.0,6990.0,Entregado,377390.0,Venta,12.0,02/09/2026,14/09/2026
4,ORD-10004,2026-09-08,NaT,Nicolás Fuentes,Araucanía,Hogar,1926800.0,6990.0,En tránsito,1933790.0,Venta,NaN,08/09/2026,NaN


### 3. Normalización de Regiones y Clientes



In [15]:
df["region"] = df["region"].astype(str).str.strip().str.lower()

diccionario = {
    "rm": "Región Metropolitana",
    "metropolitana": "Región Metropolitana",
    "region metropolitana": "Región Metropolitana",
}
df["region"] = df["region"].replace(diccionario)
df["region"] = df["region"].str.title()
df["region"] = df["region"].replace({"Valparaiso": "Valparaíso"})

df["cliente"] = df["cliente"].astype(str).str.strip().str.title()

print(df["region"].unique())
df[["cliente", "region"]].head()

<StringArray>
['Región Metropolitana',           'Valparaíso',            'Araucanía',
               'Biobío',            'Los Lagos',                'Maule',
             'Coquimbo',            'O'Higgins']
Length: 8, dtype: str


,cliente,region
0,Camila Martínez,Región Metropolitana
1,Valentina Fuentes,Valparaíso
2,Diego González,Región Metropolitana
3,Tomás Silva,Región Metropolitana
4,Nicolás Fuentes,Araucanía


### 4. Creación de KPIs y Manejo de Anomalías



In [9]:
df["ingreso_total"] = np.where(
    df["monto_venta"] > 0,
    df["monto_venta"] + df["costo_envio"],
    0,
)
df["tipo_movimiento"] = np.where(
    df["monto_venta"] > 0, "Venta", "Devolución / Ajuste"
)
df[["monto_venta", "ingreso_total", "tipo_movimiento"]].head()

,monto_venta,ingreso_total,tipo_movimiento
0,-242800.0,0.0,Devolución / Ajuste
1,3714700.0,3721690.0,Venta
2,1191800.0,1198790.0,Venta
3,370400.0,377390.0,Venta
4,1926800.0,1933790.0,Venta


### 5. Procesamiento de Fechas y Cálculo de Lead Time (Días de Entrega)



In [10]:
df["fecha_pedido"] = pd.to_datetime(
    df["fecha_pedido"], format="mixed", dayfirst=True, errors="coerce"
)
df["fecha_entrega"] = pd.to_datetime(
    df["fecha_entrega"], format="mixed", dayfirst=True, errors="coerce"
)

df["dias_entrega"] = (df["fecha_entrega"] - df["fecha_pedido"]).dt.days

df["fecha_pedido_str"] = df["fecha_pedido"].dt.strftime("%d/%m/%Y")
df["fecha_entrega_str"] = df["fecha_entrega"].dt.strftime("%d/%m/%Y")

df.head()

,pedido_id,fecha_pedido,fecha_entrega,cliente,region,categoria,monto_venta,costo_envio,estado_entrega,ingreso_total,tipo_movimiento,dias_entrega,fecha_pedido_str,fecha_entrega_str
0,ORD-10000,2026-03-07,NaT,Camila Martínez,Región Metropolitana,Vestuario,-242800.0,2990.0,NaN,0.0,Devolución / Ajuste,NaN,07/03/2026,NaN
1,ORD-10001,2026-07-09,2026-07-24,Valentina Fuentes,Valparaíso,Calzado,3714700.0,6990.0,Devuelto,3721690.0,Venta,15.0,09/07/2026,24/07/2026
2,ORD-10002,2026-08-13,2026-08-19,Diego González,Región Metropolitana,Belleza,1191800.0,6990.0,Entregado,1198790.0,Venta,6.0,13/08/2026,19/08/2026
3,ORD-10003,2026-09-02,2026-09-14,Tomás Silva,Región Metropolitana,Hogar,370400.0,6990.0,Entregado,377390.0,Venta,12.0,02/09/2026,14/09/2026
4,ORD-10004,2026-09-08,NaT,Nicolás Fuentes,Araucanía,Hogar,1926800.0,6990.0,En tránsito,1933790.0,Venta,NaN,08/09/2026,NaN


### 6. Exportación del Dataset Limpio

*(Escribe aquí tus conclusiones finales antes de exportar el archivo CSV)*

In [11]:
df.to_csv("pedidos_logistica_limpios.csv", index=False)
print("¡Archivo exportado con éxito!")

¡Archivo exportado con éxito!
